# 02. Create Views

In [0]:
dbutils.widgets.text("catalog_name", "")
dbutils.widgets.text("db_name", "")

In [0]:

catalog_name = dbutils.widgets.get("catalog_name")
db_name = dbutils.widgets.get("db_name")

spark.sql(f"USE CATALOG {catalog_name}")

spark.sql(f"USE {db_name}")

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {db_name}.vw_top_5_products_by_review AS
SELECT
    p.product_id,
    p.product_name,
    p.product_category_name,
    SUM(o.review_count) AS total_reviews,
    ROUND(SUM(o.revenue), 2) AS total_revenue
FROM {db_name}.order_tbl o
JOIN {db_name}.product p
    ON o.product_id = p.product_id
GROUP BY
    p.product_id,
    p.product_name,
    p.product_category_name
ORDER BY total_reviews DESC, total_revenue DESC
LIMIT 5
""")

display(spark.sql(f"SELECT * FROM {db_name}.vw_top_5_products_by_review"))

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {db_name}.vw_top_5_products_by_lowest_revenue AS
SELECT
    p.product_id,
    p.product_name,
    p.product_category_name,
    ROUND(SUM(o.revenue), 2) AS total_revenue,
    SUM(o.review_count) AS total_reviews
FROM {db_name}.order_tbl o
JOIN {db_name}.product p
    ON o.product_id = p.product_id
GROUP BY
    p.product_id,
    p.product_name,
    p.product_category_name
ORDER BY total_revenue ASC, total_reviews DESC
LIMIT 5
""")

display(spark.sql(f"SELECT * FROM {db_name}.vw_top_5_products_by_lowest_revenue"))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *
from pyspark.sql import Row

top_5_customers_by_revenue_df = (
    spark.table(f"{db_name}.order_tbl").alias("o")
    .join(spark.table(f"{db_name}.customer").alias("c"), on="customer_id", how="inner")
    .groupBy(
        "c.customer_id",
        "c.customer_name",
        "c.email",
        "c.city",
        "c.customer_type_name"
    )
    .agg(
        F.round(F.sum("o.revenue"), 2).alias("total_revenue"),
        F.countDistinct("o.order_id").alias("total_orders")
    )
    .orderBy(F.col("total_revenue").desc(), F.col("total_orders").desc())
    .limit(5)
)

top_5_customers_by_revenue_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{db_name}.top_5_customers_by_revenue")

display(top_5_customers_by_revenue_df)

In [0]:
print("Customer table")
display(spark.sql(f"SELECT * FROM {db_name}.customer ORDER BY customer_id"))

print("Product table")
display(spark.sql(f"SELECT * FROM {db_name}.product ORDER BY product_id"))

print("Order table")
display(spark.sql(f"SELECT * FROM {db_name}.order_tbl ORDER BY order_id"))

print("Top 5 products by review")
display(spark.sql(f"SELECT * FROM {db_name}.vw_top_5_products_by_review"))

print("Top 5 products by lowest revenue")
display(spark.sql(f"SELECT * FROM {db_name}.vw_top_5_products_by_lowest_revenue"))

print("Top 5 customers by revenue")
display(spark.sql(f"SELECT * FROM {db_name}.top_5_customers_by_revenue"))